In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
# Resolve project root on D: drive or C: drive
current_dir = Path(".").resolve()
if (current_dir / "ml").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "ml").exists():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir
PROCESSED_DIR = PROJECT_ROOT / "ml" / "data" / "employment_processed"
print(f"[*] Project Root:   {PROJECT_ROOT}")
print(f"[*] Reading Data:   {PROCESSED_DIR}")
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
with open(PROCESSED_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    label_mapping = json.load(f)
print(f"Original Train Shape: {train_df.shape}")
print(f"Original Val Shape:   {val_df.shape}")
print(f"Original Test Shape:  {test_df.shape}")
print("\nTarget Category ID for 'no_solicit_customers':", label_mapping['label2id']['no_solicit_customers'])

In [ ]:
def is_valid_employment_customer_nonsolicit(text):
    lower_text = text.lower()
    b2b_noise = ["freight", "distributor", "wholesale", "sameway", "sameday delivery", "purolator", "dynamex", "carrier", "shipment"]
    for noise in b2b_noise:
        if noise in lower_text:
            return False
    return True
def filter_split_b2b_noise(df, split_name):
    mask_target = (df["employment_category"] == "no_solicit_customers")
    target_rows = df[mask_target]
    
    valid_rows = target_rows[target_rows["clause_text"].apply(is_valid_employment_customer_nonsolicit)]
    removed_rows = target_rows[~target_rows["clause_text"].apply(is_valid_employment_customer_nonsolicit)]
    
    print(f"[{split_name.upper()}] Initial 'no_solicit_customers': {len(target_rows)} -> Retained valid employment: {len(valid_rows)}, Removed B2B noise: {len(removed_rows)}")
    
    non_target_rows = df[~mask_target]
    return pd.concat([non_target_rows, valid_rows], ignore_index=True)
train_cleaned = filter_split_b2b_noise(train_df, "train")
val_cleaned = filter_split_b2b_noise(val_df, "val")
test_cleaned = filter_split_b2b_noise(test_df, "test")

In [ ]:
handcrafted_synthetic_clauses = [
    "The Employee agrees that for a period of twelve (12) months following termination of employment, they shall not directly or indirectly solicit, canvas, or approach any customer or client of the Company.",
    "During employment and for twenty-four (24) months thereafter, the Executive will not divert or attempt to divert any active business account or client away from the Employer.",
    "The Consultant agrees that for a period of one (1) year after the separation date, they shall not accept business from or contract with any client with whom the Consultant interacted during the preceding 12 months.",
    "The Employee shall not, for eighteen (18) months following the cessation of employment for any reason, induce any customer of the Company to terminate or diminish its business relationship with the Group.",
    "During the Engagement and for twelve (12) months post-termination, the Contractor agrees not to entice away or solicit the patronage of any customer, client, or active business prospect of the Employer.",
    "For a period of two (2) years after termination, the Executive shall not, on their own behalf or on behalf of any competing business, solicit or service any clients of the Company.",
    "The Employee covenants that for 12 months post-employment, they will not directly or indirectly approach or contact any active customer of the Employer to provide competing services.",
    "During the Restricted Period of one year following termination, the Employee shall not solicit the trade or custom of any key account or active client of the Business.",
    "The Executive agrees not to solicit, entice, or encourage any client of the Employer to transfer its business to a competing firm for 24 months post-termination.",
    "For a period of 12 months after separation, the Employee agrees not to solicit or accept order for competing products from any entity that was a customer of the Company during employment.",
    "The Contractor agrees that for a period of 18 months following termination, they shall not directly or indirectly solicit the business of any customer or client of the Company.",
    "During the term of service and for one year thereafter, the Consultant shall not solicit or divert any active customer or business account from the Employer.",
    "The Employee agrees not to solicit, canvas, or service any client or customer of the Employer for a period of 12 months following the effective date of termination.",
    "For two years following termination of employment, the Executive shall not directly or indirectly contact any customer of the Company to offer competing services.",
    "The Employee covenants not to solicit, entice away, or attempt to divert any customer or client of the Employer during employment and for 12 months post-termination.",
    "During employment and for a period of 18 months following termination, the Employee agrees not to solicit the patronage of any customer or active prospect of the Company.",
    "For 12 months following separation from service, the Executive agrees not to solicit, divert, or take away any client or customer of the Employer.",
    "The Consultant agrees that for 24 months post-termination, they will not directly or indirectly solicit any customer of the Company with whom they had contact during the contract.",
    "During employment and for 12 months thereafter, the Employee shall not solicit, contact, or service any customer or business account of the Employer.",
    "The Executive covenants that for 18 months after termination, they will not solicit or induce any customer of the Company to alter its business relationship with the Firm.",
    "For a period of one year post-termination, the Employee agrees not to solicit, canvas, or approach any client of the Employer to offer competing products or services.",
    "The Contractor agrees not to solicit, divert, or accept business from any customer of the Company during the term of contract and for 12 months thereafter.",
    "During employment and for 2 years following termination, the Employee shall not directly or indirectly solicit any customer or client of the Employer.",
    "For a period of 12 months following termination, the Executive shall not solicit or service any customer or active account of the Company.",
    "The Employee agrees not to solicit, entice away, or attempt to divert any client of the Employer during employment and for 18 months post-separation.",
    "During employment and for 12 months post-termination, the Consultant shall not directly or indirectly solicit the business of any customer of the Company.",
    "For a period of 24 months following the termination of employment, the Employee covenants not to solicit or contact any customer of the Employer.",
    "The Executive agrees not to solicit, canvas, or divert any customer or client of the Company for 12 months following separation of employment.",
    "During employment and for 18 months thereafter, the Employee agrees not to solicit or accept competing business from any customer of the Employer.",
    "For 12 months post-termination, the Contractor covenants not to solicit, approach, or service any active client or customer of the Company.",
    "The Employee agrees that for 12 months following termination, they shall not directly or indirectly solicit any client or account of the Employer.",
    "During employment and for 2 years thereafter, the Executive covenants not to solicit, divert, or entice away any customer of the Business.",
    "For a period of 18 months post-termination, the Consultant agrees not to solicit or contact any customer or client of the Company.",
    "The Employee covenants not to solicit, canvas, or service any active client of the Employer during employment and for 12 months post-separation.",
    "During employment and for 12 months post-termination, the Executive shall not directly or indirectly solicit any customer of the Firm.",
    "For 24 months post-termination, the Employee agrees not to solicit, divert, or accept business from any client of the Employer.",
    "The Consultant covenants that for 12 months post-termination, they shall not solicit or approach any customer or active prospect of the Company.",
    "During employment and for 18 months post-termination, the Employee shall not directly or indirectly solicit the business of any customer of the Employer.",
    "For a period of one year post-termination, the Executive agrees not to solicit, entice away, or divert any client or account of the Company.",
    "The Contractor covenants not to solicit, canvas, or service any customer of the Employer during the engagement and for 12 months thereafter."
]
synth_records = []
for idx, clause in enumerate(handcrafted_synthetic_clauses, start=1):
    synth_records.append({
        "contract_title": f"HANDCRAFTED_EMPLOYMENT_SYNTHETIC_{idx:03d}",
        "category": "No-Solicit Of Customers",
        "clause_text": clause,
        "start_char": -1,
        "end_char": -1,
        "label_id": label_mapping['label2id']['no_solicit_customers'],
        "employment_category": "no_solicit_customers"
    })
df_synthetic = pd.DataFrame(synth_records)
print(f"[✓] Handcrafted {len(df_synthetic)} high-quality Employment Customer Non-Solicitation clauses.")

In [ ]:
from sklearn.model_selection import train_test_split
synth_train, synth_val = train_test_split(df_synthetic, test_size=0.15, random_state=42)
train_augmented = pd.concat([train_cleaned, synth_train], ignore_index=True)
val_augmented = pd.concat([val_cleaned, synth_val], ignore_index=True)
test_augmented = test_cleaned.copy() # Test set stays 100% real!
print("=== Final Augmented Dataset Shapes ===")
print(f"Train Augmented Shape: {train_augmented.shape} (Includes {len(synth_train)} synthetic clauses)")
print(f"Val Augmented Shape:   {val_augmented.shape} (Includes {len(synth_val)} synthetic clauses)")
print(f"Test Set Shape:        {test_augmented.shape} (0 synthetic clauses - 100% Real Contracts!)")
print("\n'no_solicit_customers' Clause Counts Across Splits:")
print(f" - Train: {sum(train_augmented['employment_category'] == 'no_solicit_customers')}")
print(f" - Val:   {sum(val_augmented['employment_category'] == 'no_solicit_customers')}")
print(f" - Test:  {sum(test_augmented['employment_category'] == 'no_solicit_customers')}")

In [ ]:
AUGMENTED_DIR = PROJECT_ROOT / "ml" / "data" / "employment_augmented"
AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)
train_augmented.to_csv(AUGMENTED_DIR / "train.csv", index=False)
val_augmented.to_csv(AUGMENTED_DIR / "val.csv", index=False)
test_augmented.to_csv(AUGMENTED_DIR / "test.csv", index=False)
with open(AUGMENTED_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2)
print(f"[✓] Successfully exported augmented dataset to: {AUGMENTED_DIR.resolve()}")
print("Exported Artifacts:")
print(" - train.csv")
print(" - val.csv")
print(" - test.csv")
print(" - label_mapping.json")